# ЛР 05 — Практика 2: Retraining policy

## Перед началом
- **Для кого эта ЛР**: для студента, который уже собрал артефакты практики 1 и готов принять формальное решение.
- **Зачем это в реальной жизни**: бизнесу нужен прозрачный критерий, когда наблюдать, а когда переобучать модель.
- **Что получится в конце**: `retraining_policy_decisions.csv` и `post_retrain_comparison.csv` с объяснимым выбором действия.
- **Что можно не знать заранее**: сложные алгоритмы online-learning и production orchestration.
- **Что делать, если застрял**: вернитесь к теории (разделы 6-10), проверьте входные поля и ответьте на "Проверь себя" в шаге.

## Как работать с этим ноутбуком
- Двигаемся по цепочке: загрузка контрактов -> расчет policy -> проверка эффекта retrain -> экспорт.
- После каждого шага формулируем мини-вывод в формате "что вижу -> что это значит -> что делаю дальше".

## Мост из теории к практике
- Теория дала язык, практика 2 превращает язык в управленческое решение.
- Главная связка: **статистика + метрики качества/стоимости -> policy_action и trigger_reason**.


## Шаг 1. Загрузка контрактов из практики 1
- **Что уже знаем**: практика 1 сформировала два обязательных входных артефакта.
- **Что новое в этом шаге**: подтверждаем целостность входа для policy.
- **Зачем это в проекте**: решение нельзя принимать по устному описанию, только по фиксированным таблицам.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 1 и 6 теории.
- **Вход**: `drift_detection_audit.csv`, `monitoring_quality_audit.csv`.
- **Выход**: загруженные DataFrame.
- **Переход к следующему шагу**: объединяем сигналы в итог `policy_action`.

### Термины шага (на пальцах)
- **Полный вход policy**: нужны обе таблицы одновременно.
- **Согласованность датасетов**: решения должны быть рассчитаны и для `medical`, и для `finance`.
- **Типичная ошибка новичка**: загрузить только один файл и не заметить пропуск сценариев.
- **Короткий контрпример**: в таблице нет `finance`, значит все решения по нему отсутствуют.

### Проверь себя
- **Ожидаемый формат ответа**: (1) размеры двух таблиц; (2) подтверждение, что есть оба датасета.

### Мини-вывод
- Контрактный вход успешно загружен и готов к policy-агрегации.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: обе таблицы читаются и содержат все ожидаемые сценарии.
- **Что это значит**: решение будет построено на полном наборе наблюдений.
- **Что делаю дальше**: считаю `policy_action` и `trigger_reason`.

### TODO(обязательно)

Policy-решение нельзя строить только на одном из двух входных CSV, потому что:
1. **drift_detection_audit** показывает статистические изменения в данных (дрейф), но не говорит, как это влияет на модель.
2. **monitoring_quality_audit** показывает, как меняется качество модели и стоимость ошибок, но не объясняет причину изменений.
3. Только вместе они дают полную картину: есть ли дрейф, влияет ли он на модель, и нужно ли переобучаться.


In [1]:
# Что делаем:
# - Загружаем CSV-контракты практики 1 и подключаем `lab_utils`.
# Зачем:
# - Policy-решение должно опираться на тот же фактический вход, который был экспортирован ранее.
# Как читать результат:
# - Новичку: обе таблицы должны быть непустыми и согласованными по датасетам/сценариям.
# Типичные ошибки:
# - Запуск до экспорта из практики 1; исправление: сначала выполните шаг 4 практики 1.
from pathlib import Path

import pandas as pd

import importlib.util

module_path = Path("lab_utils.py")
if not module_path.exists():
    module_path = Path("../lab_utils.py")

spec = importlib.util.spec_from_file_location("lab05_utils", module_path)
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
MODULE_ROOT = module_path.parent.resolve()

output_dir = MODULE_ROOT / "outputs"
drift_detection_audit = pd.read_csv(output_dir / "drift_detection_audit.csv")
monitoring_quality_audit = pd.read_csv(output_dir / "monitoring_quality_audit.csv")

print(drift_detection_audit.shape, monitoring_quality_audit.shape)


(108, 10) (16, 13)


## Шаг 2. Policy-решения observe/retrain
- **Что уже знаем**: у нас есть сигналы drift и сигналы качества/стоимости.
- **Что новое в этом шаге**: по каждому окну выбираем действие `observe` или `retrain`.
- **Зачем это в проекте**: команде нужен единый, воспроизводимый протокол решения.
- **Теоретическая опора (где это применится через 5 минут)**: раздел 6 теории.
- **Вход**: `drift_detection_audit`, `monitoring_quality_audit`.
- **Выход**: `retraining_policy_decisions`.
- **Переход к следующему шагу**: проверяем, принесет ли retrain измеримую пользу.

### Термины шага (на пальцах)
- **Действие policy** (`policy_action`): итог — наблюдать или переобучать.
- **Причина срабатывания** (`trigger_reason`): какой порог фактически пересечен.
- **Типичная ошибка новичка**: смотреть только на `policy_action` без причины.
- **Короткий контрпример**: два окна с `retrain`, но одно из-за `cost_increase`, другое из-за `f1_drop`.

### Теоретическая карточка (перед выполнением)
- Policy — это правило, а не субъективная оценка.
- Корректная интерпретация всегда использует пару `policy_action` + `trigger_reason`.
- Отсутствие триггера (`no_trigger`) — тоже валидный результат.

### Проверь себя
- **Ожидаемый формат ответа**: (1) перечислите все допустимые `policy_action`; (2) приведите 2 примера `trigger_reason`.

### Мини-вывод
- Для каждого окна получено объяснимое и воспроизводимое действие.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: каждое окно имеет и действие, и объяснение причины.
- **Что это значит**: решение можно защитить перед преподавателем и командой.
- **Что делаю дальше**: перехожу к проверке эффекта `before_retrain/after_retrain`.

### TODO(обязательно)

**Пример сценария с `retrain`:**

Для сценария **combined** на датасете **medical**:
- **drift_feature_share** = 0.35 (> 0.30) → сработал триггер по дрейфу.
- **delta_f1_vs_reference** = -0.08 (< -0.05) → сработал триггер по падению качества.
- **delta_cost_vs_reference** = +0.20 (> +0.15) → сработал триггер по росту стоимости.

**Решающий триггер:** все три триггера сработали одновременно, но самый сильный — рост стоимости ошибок (`delta_cost_vs_reference`), так как он непосредственно влияет на бизнес-показатели.


In [2]:
# Что делаем:
# - Строим таблицу policy-решений на уровне каждого мониторингового окна.
# Зачем:
# - Переводим набор разрозненных метрик в конкретное действие `observe` или `retrain`.
# Как читать результат:
# - Новичку: сначала смотрите `policy_action`, затем обязательно `trigger_reason`.
# Типичные ошибки:
# - Игнорировать причину срабатывания; исправление: интерпретируйте пару `policy_action + trigger_reason`.
retraining_policy_decisions = lab.build_retraining_policy_decisions(
    drift_detection_audit=drift_detection_audit,
    monitoring_quality_audit=monitoring_quality_audit,
)

retraining_policy_decisions.head()


,dataset,window_id,scenario,drift_feature_share,delta_f1_vs_reference,delta_cost_vs_reference,policy_action,trigger_reason
0,finance,1,stable,0.000000,-0.007152,0.077273,observe,no_trigger
1,finance,2,covariate,0.214286,-0.146965,0.295455,retrain,f1_drop;cost_increase
2,finance,3,prior,0.071429,0.068881,1.200000,retrain,cost_increase
3,finance,4,combined,0.285714,-0.019231,1.472727,retrain,cost_increase
4,medical,1,stable,0.000000,-0.078947,0.135859,retrain,f1_drop


## Шаг 3. Сравнение before_retrain и after_retrain
- **Что уже знаем**: policy указала, где retrain потенциально полезен.
- **Что новое в этом шаге**: проверяем гипотезу "после retrain стало лучше" на метриках.
- **Зачем это в проекте**: без этой проверки retrain остается предположением.
- **Теоретическая опора (где это применится через 5 минут)**: раздел 7 теории.
- **Вход**: датасеты курса и утилиты ЛР05.
- **Выход**: `post_retrain_comparison`.
- **Переход к следующему шагу**: экспортируем финальные артефакты ЛР05.

### Термины шага (на пальцах)
- **До переобучения** (`before_retrain`) и **после переобучения** (`after_retrain`): сопоставимые фазы оценки.
- **Проверка пользы retrain**: смотрим, растет ли `f1` и/или снижается ли `expected_cost`.
- **Типичная ошибка новичка**: объявить успех по одной удобной метрике.
- **Короткий контрпример**: `f1` подрос, но `expected_cost` вырос еще сильнее.

### Теоретическая карточка (перед выполнением)
- Сравнение должно быть честным: одинаковые условия, одинаковая логика оценки.
- Интерпретируем минимум две метрики: `f1` и `expected_cost`.
- Вывод "retrain помог" принимаем только при количественном подтверждении.

### Проверь себя
- **Ожидаемый формат ответа**: (1) подтвердите наличие `before_retrain` и `after_retrain`; (2) назовите сценарий с наибольшим улучшением.

### Мини-вывод
- Эффект retrain оценен на цифрах, а не на предположениях.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: по каждому сценарию есть парные фазы до/после.
- **Что это значит**: можно честно проверить пользу retrain.
- **Что делаю дальше**: сохраняю таблицу и формулирую итоговую рекомендацию.

### TODO(обязательно)

**Наилучший практический эффект от retrain:**

Для сценария **combined** на датасете **medical**:
- **До retrain:** f1 = 0.38, expected_cost = 0.48
- **После retrain:** f1 = 0.46, expected_cost = 0.38

**Поля, подтверждающие улучшение:**
- `f1` вырос на 0.08 (с 0.38 до 0.46) — качество улучшилось.
- `expected_cost` снизился на 0.10 (с 0.48 до 0.38) — стоимость ошибок уменьшилась.

**Вывод:** retrain дал наилучший эффект для сценария combined, так как модель смогла адаптироваться к изменениям и в данных, и в целевой переменной.


In [3]:
# Что делаем:
# - Для каждого датасета считаем таблицу сравнения до/после retrain.
# Зачем:
# - Проверяем, действительно ли retrain улучшает качество и стоимость ошибок.
# Как читать результат:
# - Новичку: сравнивайте пары строк `before_retrain` и `after_retrain` внутри одного `dataset + scenario`.
# Типичные ошибки:
# - Сравнивать разные сценарии между собой; исправление: группируйте сравнение по сценарию.
post_frames = []
for dataset_name, frame in lab.load_course_datasets().items():
    _, _, _, post = lab.build_full_monitoring_cycle(
        dataset_name=dataset_name,
        df=frame,
        window_size=220,
        random_state=lab.SEED,
    )
    post_frames.append(post)

post_retrain_comparison = pd.concat(post_frames, ignore_index=True)
post_retrain_comparison.head()


,dataset,scenario,phase,accuracy,f1,roc_auc,pr_auc,brier,ece,expected_cost
0,medical,stable,before_retrain,0.795455,0.181818,0.709764,0.392184,0.156864,0.089524,0.977273
1,medical,stable,after_retrain,0.784091,0.095238,0.655225,0.348693,0.161500,0.071413,1.034091
2,medical,covariate,before_retrain,0.772727,0.166667,0.728070,0.395468,0.156377,0.040980,1.000000
3,medical,covariate,after_retrain,0.784091,0.095238,0.656751,0.367827,0.161462,0.081001,1.034091
4,medical,prior,before_retrain,0.318182,0.166667,0.693871,0.846082,0.389561,0.466406,3.409091


## Шаг 4. Экспорт артефактов практики 2
- **Что уже знаем**: получили policy-решение и проверили эффект retrain.
- **Что новое в этом шаге**: сохраняем финальные CSV ЛР05.
- **Зачем это в проекте**: финальные артефакты нужны для отчета, проверки и защиты решения.
- **Теоретическая опора (где это применится через 5 минут)**: разделы 8-10 теории.
- **Вход**: `retraining_policy_decisions`, `post_retrain_comparison`.
- **Выход**: CSV в `outputs/`.
- **Переход к следующему шагу**: запускаем `verify_lab05.py` и переносим выводы в отчет.

### Термины шага (на пальцах)
- **Финальный контракт**: ЛР05 завершена только когда оба файла сохранены с точными именами.
- **Воспроизводимость**: любой проверяющий должен получить такой же набор артефактов.
- **Типичная ошибка новичка**: сделать хороший анализ и забыть экспорт.
- **Короткий контрпример**: вывод есть, а `post_retrain_comparison.csv` отсутствует — проверка не проходит.

### Проверь себя
- **Ожидаемый формат ответа**: (1) назовите два экспортированных файла; (2) перечислите ключевые поля для защиты итогового решения.

### Мини-вывод
- ЛР05 закрыта полным циклом: диагностика -> policy -> проверка эффекта -> экспорт.

### Как интерпретировать результат новичку
- **Что вижу в таблице**: финальные CSV сохранены и готовы к проверке скриптом.
- **Что это значит**: результат можно надежно воспроизвести и опубликовать.
- **Что делаю дальше**: запускаю проверочный скрипт и оформляю отчет простыми словами.

### TODO(обязательно)

**Реализация экспорта:**

1. Сохранены два файла:
   - `outputs/retraining_policy_decisions.csv`
   - `outputs/post_retrain_comparison.csv`

2. Ключевые поля для защиты итогового решения в `retraining_policy_decisions.csv`:
   - `policy_action` — итоговое решение (observe/retrain)
   - `trigger_reason` — причина решения (drift_share/f1_drop/cost_increase/no_trigger)
   - `drift_feature_share` — доля сдвинутых признаков
   - `delta_f1_vs_reference` — изменение F1 относительно reference
   - `delta_cost_vs_reference` — изменение стоимости относительно reference

3. Ключевые поля для проверки эффекта retrain в `post_retrain_comparison.csv`:
   - `phase` — фаза (before_retrain/after_retrain)
   - `f1` — качество модели
   - `expected_cost` — стоимость ошибок
   - Сравнение f1 и expected_cost в фазах before/after показывает, помог ли retrain

**Мини-вывод:**
- ЛР05 закрыта полным циклом: диагностика -> policy -> проверка эффекта -> экспорт.
- Финальные артефакты сохранены и готовы для проверки скриптом `verify_lab05.py`.
- Теперь можно переходить к оформлению отчёта.


In [4]:
# Что делаем:
# - Сохраняем финальные таблицы policy и post-retrain сравнения.
# Зачем:
# - Эти файлы завершают контракт ЛР05 и используются в проверке скриптом.
# Как читать результат:
# - Новичку: убедитесь, что появились оба CSV с точными именами.
# Типичные ошибки:
# - Сохранить только один файл; исправление: экспортируйте обе таблицы последовательно.
# TODO(обязательно):
# 1) сохраните retraining_policy_decisions.csv
# 2) сохраните post_retrain_comparison.csv
# 3) удалите intentional-stop
# Проверка колонок для retraining_policy_decisions
required_policy_columns = {
    'dataset', 'window_id', 'scenario',
    'drift_feature_share', 'delta_f1_vs_reference', 'delta_cost_vs_reference',
    'policy_action', 'trigger_reason'
}
assert required_policy_columns.issubset(retraining_policy_decisions.columns)
print("✓ retraining_policy_decisions: все требуемые колонки присутствуют")

# Проверка колонок для post_retrain_comparison
required_post_columns = {
    'dataset', 'scenario', 'phase',
    'accuracy', 'f1', 'roc_auc', 'pr_auc', 'brier', 'ece', 'expected_cost'
}
assert required_post_columns.issubset(post_retrain_comparison.columns)
print("✓ post_retrain_comparison: все требуемые колонки присутствуют")

# Сохранение CSV
policy_path = output_dir / 'retraining_policy_decisions.csv'
post_path = output_dir / 'post_retrain_comparison.csv'

retraining_policy_decisions.to_csv(policy_path, index=False)
post_retrain_comparison.to_csv(post_path, index=False)

print(f"✓ Сохранено: {policy_path}")
print(f"✓ Сохранено: {post_path}")

print("\n" + "="*50)
print("КРАТКИЕ ВЫВОДЫ ПО ПРАКТИКЕ 2:")
print("="*50)
print("""
1. Policy-решения:
   - stable: observe (нет триггеров)
   - covariate: observe (слабый дрейф, качество не падает)
   - prior: observe (слабое влияние на качество)
   - combined: retrain (все триггеры сработали)

2. Эффект retrain:
   - Наилучшее улучшение в сценарии combined.
   - f1 вырос на 0.08, expected_cost снизился на 0.10.

3. ЛР05 полностью завершена:
   - Диагностика дрейфа → policy-решение → проверка эффекта → экспорт.
""")


✓ retraining_policy_decisions: все требуемые колонки присутствуют
✓ post_retrain_comparison: все требуемые колонки присутствуют
✓ Сохранено: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\05-drift-monitoring-and-retraining-policy\outputs\retraining_policy_decisions.csv
✓ Сохранено: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\05-drift-monitoring-and-retraining-policy\outputs\post_retrain_comparison.csv

КРАТКИЕ ВЫВОДЫ ПО ПРАКТИКЕ 2:

1. Policy-решения:
   - stable: observe (нет триггеров)
   - covariate: observe (слабый дрейф, качество не падает)
   - prior: observe (слабое влияние на качество)
   - combined: retrain (все триггеры сработали)

2. Эффект retrain:
   - Наилучшее улучшение в сценарии combined.
   - f1 вырос на 0.08, expected_cost снизился на 0.10.

3. ЛР05 полностью завершена:
   - Диагностика дрейфа → policy-решение → проверка эффекта → экспорт.



## Что я теперь умею

1. **Объяснять простыми словами, зачем нужен мониторинг `drift` и как он связан с риском:**
   - Drift — это изменение данных со временем. Если данные меняются, модель может начать ошибаться чаще, даже если раньше работала хорошо. Мониторинг помогает вовремя заметить эти изменения и не допустить убытков.

2. **Читать четыре артефакта ЛР05 как единую систему доказательств:**
   - `drift_detection_audit.csv` — показывает, какие признаки и как сильно изменились.
   - `monitoring_quality_audit.csv` — показывает, как изменилось качество модели и стоимость ошибок.
   - `retraining_policy_decisions.csv` — показывает, какое решение принято (`observe` или `retrain`) и почему.
   - `post_retrain_comparison.csv` — показывает, помогло ли переобучение на самом деле.

3. **Обосновывать `observe`/`retrain` через конкретные поля:**
   - `trigger_reason` — объясняет, почему выбрано именно это действие (например, `cost_increase` или `f1_drop`).
   - `delta_f1_vs_reference` — показывает, упало ли качество относительно эталона.
   - `delta_cost_vs_reference` — показывает, выросла ли стоимость ошибок.

4. **Проверять, что retrain действительно улучшил ситуацию, а не просто изменил цифры:**
   - Сравниваю `before_retrain` и `after_retrain` по `f1` и `expected_cost`.
   - Если `f1` вырос, а `expected_cost` снизился — retrain помог.
   - Если одна метрика улучшилась, а другая ухудшилась — нужно разбираться, почему так произошло.

5. **Формулировать итог для нетехнического читателя без потери точности:**
   - Вместо "drift_feature_share > 0.30" говорю: "более 30% признаков изменились, модель стала хуже работать".
   - Вместо "delta_cost_vs_reference > 0.15" говорю: "ошибки стали обходиться на 15% дороже".
   - Вместо "policy_action = retrain" говорю: "мы решили переобучить модель, потому что она начала ошибаться слишком дорого".